# 2.3 Survive the channel

2.3 Survive the channel: evaluate the block-mean QIM design (embed_robust
/ extract_robust) and naive LSB against additive Gaussian noise and JPEG-75,
using the supplied channel functions unchanged. Shows the BER-vs-sigma plot
and prints the results table.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
sys.path.insert(0, str(Path.cwd()))
from starter import (embed_lsb, extract_lsb, embed_robust, extract_robust,
                      degrade_gaussian, degrade_jpeg, ber, psnr)

In [ ]:
ROOT = Path.cwd().parents[1]

cover = np.asarray(Image.open(ROOT / "images/p2/cover_textured.png"))
N_BITS = 128
SIGMAS = [0, 1, 2, 5, 10, 20]
BLOCK_SIZE = 16
DELTA = 2.0

In [ ]:
rng = np.random.default_rng(7)
bits = rng.integers(0, 2, size=N_BITS, dtype=np.uint8)

In [ ]:
# Naive LSB

In [ ]:
stego_naive = embed_lsb(cover, bits, plane=0)
psnr_naive = psnr(stego_naive, cover)
print(f"naive LSB stego PSNR = {psnr_naive:.2f} dB")

In [ ]:
ber_naive = []
for sigma in SIGMAS:
    degraded = degrade_gaussian(stego_naive, sigma, rng=np.random.default_rng(1000 + sigma))
    rec = extract_lsb(degraded, N_BITS, plane=0)
    b = ber(rec, bits)
    ber_naive.append(b)
    print(f"naive LSB sigma={sigma:3d}: BER={b:.4f}")

In [ ]:
jpeg_naive_out = degrade_jpeg(stego_naive, 75)
jpeg_naive_bits = extract_lsb(jpeg_naive_out, N_BITS, plane=0)
ber_naive_jpeg = ber(jpeg_naive_bits, bits)
print(f"naive LSB JPEG75: BER={ber_naive_jpeg:.4f}")

In [ ]:
# Robust design (block-mean QIM, delta=2, block=16, 32x repetition)

In [ ]:
stego_robust = embed_robust(cover, bits, block_size=BLOCK_SIZE, delta=DELTA)
psnr_robust = psnr(stego_robust, cover)
print(f"\nrobust stego PSNR = {psnr_robust:.2f} dB")

In [ ]:
ber_robust = []
for sigma in SIGMAS:
    degraded = degrade_gaussian(stego_robust, sigma, rng=np.random.default_rng(2000 + sigma))
    rec = extract_robust(degraded, cover, N_BITS, block_size=BLOCK_SIZE)
    b = ber(rec, bits)
    ber_robust.append(b)
    print(f"robust sigma={sigma:3d}: BER={b:.4f}")

In [ ]:
jpeg_robust_out = degrade_jpeg(stego_robust, 75)
jpeg_robust_bits = extract_robust(jpeg_robust_out, cover, N_BITS, block_size=BLOCK_SIZE)
ber_robust_jpeg = ber(jpeg_robust_bits, bits)
print(f"robust JPEG75: BER={ber_robust_jpeg:.4f}")

In [ ]:
# zero-error requirement check
assert ber_robust[SIGMAS.index(5)] == 0.0, "FAILS zero-error requirement at sigma=5"
assert ber_robust_jpeg == 0.0, "FAILS zero-error requirement at JPEG75"
print("\nZero-error requirement (sigma=5 AND JPEG75): PASSED")

In [ ]:
# distortion budget per modified pixel
n_pixels_modified = cover.size  # every pixel shifted by +-delta
total_sq_error = n_pixels_modified * DELTA**2
print(f"\nfraction of pixels modified: {n_pixels_modified/cover.size*100:.1f}% "
      f"(every pixel, by exactly +-{DELTA} grey levels)")

In [ ]:
# BER vs sigma plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(SIGMAS, ber_naive, "o-", label="naive LSB", color="indianred")
ax.plot(SIGMAS, ber_robust, "s-", label=f"robust (block QIM, $\\delta$={DELTA})", color="steelblue")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1, label="chance (BER=0.5)")
ax.axvline(5, color="green", linestyle="--", linewidth=1, alpha=0.6, label="required $\\sigma=5$")
ax.set_xlabel("$\\sigma$ (Gaussian noise)")
ax.set_ylabel("Bit error rate")
ax.set_title("BER against $\\sigma$: naive LSB vs. designed scheme")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# results table

In [ ]:
header = f"{'Scheme':16s} {'PSNR':>8s} " + " ".join(f"s={s:>5}" for s in SIGMAS) + "   JPEG75"
print(header)
print(f"{'Naive LSB':16s} {psnr_naive:7.2f}dB " +
      " ".join(f"{b:6.3f}" for b in ber_naive) + f"  {ber_naive_jpeg:6.3f}")
print(f"{'Robust (ours)':16s} {psnr_robust:7.2f}dB " +
      " ".join(f"{b:6.3f}" for b in ber_robust) + f"  {ber_robust_jpeg:6.3f}")